# 03 - CV Training: ResNet50 on HAM10000
**ZHAW AI-Applications: Skin Lesion Classifier**

**Optimized for Google Colab with T4/A100 GPU**

This notebook:
1. Downloads HAM10000 via Kaggle API
2. Fine-tunes ResNet50 (transfer learning)
3. Trains for 20 epochs with class-weighted loss
4. Plots confusion matrix + F1 per class
5. Visualizes Grad-CAM explanations

**Target: 75-85% Accuracy, Macro F1 >= 0.65**

In [ ]:
# Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q kaggle sentence-transformers anthropic gradio albumentations grad-cam shap xgboost
print('Dependencies installed')

In [ ]:
# Mount Google Drive (to save models)
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
import subprocess
result = subprocess.run(['git', 'clone', 'https://github.com/Jeff0559/skin-lesion-classifier.git'],
                       capture_output=True, text=True)
print(result.stdout or result.stderr)
%cd skin-lesion-classifier

In [ ]:
# Set up environment
import os
from pathlib import Path

# Set API keys (use Colab Secrets or paste directly)
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Or paste directly (do NOT commit):
os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME'  # <-- Change this
os.environ['KAGGLE_KEY'] = 'YOUR_KEY'            # <-- Change this
os.environ['DEVICE'] = 'cuda'

# Create .env file
with open('.env', 'w') as f:
    f.write(f'KAGGLE_USERNAME={os.environ["KAGGLE_USERNAME"]}\n')
    f.write(f'KAGGLE_KEY={os.environ["KAGGLE_KEY"]}\n')
    f.write('DEVICE=cuda\n')
print('Environment configured')

In [ ]:
# Download and prepare HAM10000
import subprocess
result = subprocess.run(['python', '-m', 'src.data.prepare_ham10000'],
                       capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

## Training Configuration

In [ ]:
import sys
sys.path.insert(0, '.')
from src.config import CV_CONFIG, CLASS_NAMES, NUM_CLASSES

# Adjust config for Colab
CV_CONFIG['num_epochs'] = 20
CV_CONFIG['batch_size'] = 32  # Increase if GPU allows: 64 for A100
CV_CONFIG['lr'] = 1e-4

print('Training config:')
for k, v in CV_CONFIG.items():
    print(f'  {k}: {v}')

## Train Model

In [ ]:
from src.cv.train import train

results = train(config={
    'device': 'cuda',
    'num_epochs': 20,
    'batch_size': 32,
})

print(f"\nBest val F1: {max(results['history']['val_f1']):.4f}")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt
import json

with open('logs/cv_results.json') as f:
    res = json.load(f)

history = res['history']
epochs = range(1, len(history['train_loss'])+1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'], label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(epochs, history['val_f1'], label='Val F1', color='green')
axes[2].set_title('Val Macro F1')
axes[2].legend()

plt.tight_layout()
plt.savefig('docs/screenshots/training_curves.png', dpi=150)
plt.show()

## Confusion Matrix

In [ ]:
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix
from src.cv.model import load_model
from src.cv.preprocessing import get_dataloaders
from src.config import CLASS_NAMES, HAM10000_CLASSES

# Load best model
model = load_model('models/resnet50_best.pth', device='cuda')
loaders = get_dataloaders()

# Collect predictions
all_preds, all_labels = [], []
model.eval()
import torch
with torch.no_grad():
    for imgs, labels in loaders['test']:
        imgs = imgs.cuda()
        preds = model(imgs).argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (absolute)')

sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix (normalized)')

plt.tight_layout()
plt.savefig('docs/screenshots/confusion_matrix.png', dpi=150)
plt.show()

## F1 per Class

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

## Grad-CAM Visualization

In [ ]:
from src.cv.grad_cam import GradCAM, overlay_heatmap
from PIL import Image
import numpy as np
import pandas as pd

test_df = pd.read_csv('data/processed/test.csv')

fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(12, 3*len(CLASS_NAMES)))
gradcam  = GradCAM(model)

for i, cls in enumerate(CLASS_NAMES):
    sample = test_df[test_df['label_name']==cls].sample(1, random_state=42)
    row = sample.iloc[0]
    img_pil = Image.open(row['image_path']).convert('RGB')
    heatmap, pred_idx, conf = gradcam(img_pil)
    overlay = overlay_heatmap(img_pil, heatmap)

    axes[i][0].imshow(img_pil)
    axes[i][0].set_title(f'{cls} (true)', fontsize=8)
    axes[i][0].axis('off')

    axes[i][1].imshow(heatmap, cmap='jet')
    axes[i][1].set_title(f'Grad-CAM', fontsize=8)
    axes[i][1].axis('off')

    axes[i][2].imshow(overlay)
    axes[i][2].set_title(f'Pred: {CLASS_NAMES[pred_idx]} ({conf:.0%})', fontsize=8)
    axes[i][2].axis('off')

plt.suptitle('Grad-CAM per Class', fontsize=14)
plt.tight_layout()
plt.savefig('docs/screenshots/grad_cam.png', dpi=100)
plt.show()

## Save Model to Google Drive

In [ ]:
import shutil
drive_path = '/content/drive/MyDrive/ZHAW_SkinClassifier/'
Path(drive_path).mkdir(parents=True, exist_ok=True)
shutil.copy('models/resnet50_best.pth', drive_path)
print(f'Model saved to Google Drive: {drive_path}resnet50_best.pth')